### LLM LAB
Objective:
Adapt a pre-trained open-weight language model to a specialized domain using Parameter-Efficient Fine-Tuning (LoRA - Low-Rank Adaptation) 

In [1]:
# Run this cell once to install required libraries on Windows
!pip install -q torch transformers accelerate

## Load the Tokenizer & Model Locally

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# 1. Define the free, lightweight open-weight model
model_id = "HuggingFaceTB/SmolLM2-135M-Instruct"

# 2. Load the BPE Tokenizer and Causal LM weights locally
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.float32,   # Safe default for Windows CPU/laptops
    device_map="auto"
)

# 3. Verify hardware device and inspect the Transformer architecture
print(f"Running on device: {model.device}")
print(f"Total parameters: {model.num_parameters():,}")
print("\n--- Model Architecture ---")
print(model)

config.json:   0%|          | 0.00/861 [00:00<?, ?B/s]

C:\Users\phdst\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\phdst\.cache\huggingface\hub\models--HuggingFaceTB--SmolLM2-135M-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/3.76k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

... (3 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Running on device: cpu
Total parameters: 134,515,008

--- Model Architecture ---
LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(49152, 576, padding_idx=2)
    (layers): ModuleList(
      (0-29): 30 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=576, out_features=576, bias=False)
          (k_proj): Linear(in_features=576, out_features=192, bias=False)
          (v_proj): Linear(in_features=576, out_features=192, bias=False)
          (o_proj): Linear(in_features=576, out_features=576, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=576, out_features=1536, bias=False)
          (up_proj): Linear(in_features=576, out_features=1536, bias=False)
          (down_proj): Linear(in_features=1536, out_features=576, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((576,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((5

## Next-Token Probability Prediction

In [2]:
import torch
import torch.nn.functional as F

# 1. Define a starting text prompt
prompt = "The capital of France is"

# 2. Tokenize the prompt into token IDs (integers)
input_ids = tokenizer.encode(prompt, return_tensors="pt")
print(f"Input Prompt: '{prompt}'")
print(f"Token IDs: {input_ids.tolist()}")

# 3. Forward pass through the model (No gradient needed for inference)
with torch.no_grad():
    outputs = model(input_ids)

# 4. Extract logits for the LAST token in the sequence
# outputs.logits shape is (batch_size, sequence_length, vocab_size)
next_token_logits = outputs.logits[0, -1, :] 

# 5. Convert raw logits into probabilities using Softmax
probabilities = F.softmax(next_token_logits, dim=-1)

# 6. Find the top 5 most likely next tokens
top_k = 5
top_probs, top_indices = torch.topk(probabilities, top_k)

print(f"\nTop {top_k} Predicted Next Tokens:")
for i in range(top_k):
    token_id = top_indices[i].item()
    token_text = tokenizer.decode([token_id])
    prob = top_probs[i].item() * 100
    print(f"  {i+1}. '{token_text}' (ID: {token_id}) -> {prob:.2f}% probability")

Input Prompt: 'The capital of France is'
Token IDs: [[504, 3575, 282, 4649, 314]]

Top 5 Predicted Next Tokens:
  1. ' Paris' (ID: 7042) -> 46.93% probability
  2. ' the' (ID: 260) -> 26.57% probability
  3. ' located' (ID: 3807) -> 8.32% probability
  4. ' called' (ID: 1217) -> 1.75% probability
  5. ' Le' (ID: 2250) -> 1.49% probability


 ## Decoding Strategies (Temperature, Top-$k$, and Top-$p$)

In [3]:
## Now let's move from predicting a single token to generating full sequences, and see how decoding hyperparameters 
## reshape that probability distribution at every step.

In [4]:
# Prompt designed to highlight differences between deterministic and creative sampling
prompt = "In the year 2050, artificial intelligence will"
inputs = tokenizer(prompt, return_tensors="pt")

def generate_text(label, do_sample, temperature=1.0, top_k=50, top_p=1.0):
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=35,
            do_sample=do_sample,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
            pad_token_id=tokenizer.eos_token_id
        )
    generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    print(f"--- {label} ---")
    print(generated_text)
    print()

# 1. Greedy Decoding (Always picks highest probability token - deterministic)
generate_text("1. Greedy Decoding (Deterministic)", do_sample=False)

# 2. Balanced Nucleus Sampling (Standard Chatbot Setting)
generate_text("2. Balanced Sampling (Temp=0.7, Top-p=0.9)", do_sample=True, temperature=0.7, top_p=0.9)

# 3. High Temperature Sampling (Flattens distribution - high creativity/randomness)
generate_text("3. High Creativity / Chaos (Temp=1.5, Top-k=50)", do_sample=True, temperature=1.5, top_k=50)

--- 1. Greedy Decoding (Deterministic) ---
In the year 2050, artificial intelligence will have surpassed human intelligence, and the world will be a world where AI is the dominant force.

In this scenario, the AI will have developed a form of consciousness that

--- 2. Balanced Sampling (Temp=0.7, Top-p=0.9) ---
In the year 2050, artificial intelligence will have surpassed human intelligence in many areas, and it will be able to make decisions without being influenced by emotions.

This means that the future will see a significant shift towards

--- 3. High Creativity / Chaos (Temp=1.5, Top-k=50) ---
In the year 2050, artificial intelligence will take a significant
centenario where there has now been no significant increase in cyberattacks targeting critical resources. This, we
can say that AI, whether it exists by other



##### Greedy Decoding (do_sample=False):Always picks the single highest-probability token ($\arg\max$).
Notice the subtle repetition loop starting to form: "the world will be a world where..."
—a classic artifact of greedy decoding in smaller models because high-frequency phrases reinforce themselves.

#### Balanced Nucleus Sampling (Temp=0.7, Top-p=0.9):Scaling logits by $T = 0.7$ sharpens the probability distribution slightly toward plausible words,
while Top-$p$ (0.9) dynamically restricts sampling to the smallest set of tokens whose cumulative probability exceeds 90%.R
Result: Fluent, coherent, and natural phrasing ("make decisions without being influenced by emotions") without falling into repetitive loops.
    
#### High Temperature (Temp=1.5, Top-k=50):Dividing logits by $T = 1.5$ flattens the distribution, 
boosting low-probability tail tokens. Because top_k=50 still allows the 50th most likely token to be chosen with non-trivial probability,
the model suddenly jumps to an out-of-place token ("centenario") and loses grammatical coherence.

## Build a Mini Interactive Chatbot Application

Now let's wrap everything together into Goal 4: turning this local model into an interactive multi-turn Chat Application

In [ ]:
# Initialize conversation history with a System Prompt
chat_history = [
    {"role": "system", "content": "You are a helpful, concise computer science teaching assistant."}
]

print("🤖 Local LLM Chatbot Ready! (Type 'exit' or 'quit' to stop)\n")

while True:
    # 1. Get user input from Jupyter notebook input box
    user_input = input("You: ")
    if user_input.strip().lower() in ["exit", "quit"]:
        print("Chatbot session ended.")
        break
    if not user_input.strip():
        continue

    # 2. Append user message to conversation history
    chat_history.append({"role": "user", "content": user_input})

    # 3. Format the entire conversation using the model's Chat Template
    formatted_prompt = tokenizer.apply_chat_template(
        chat_history,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = tokenizer(formatted_prompt, return_tensors="pt")

    # 4. Generate response using balanced decoding settings
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )

    # 5. Slice off the input prompt tokens so we only decode the newly generated reply
    new_tokens = output_ids[0][inputs["input_ids"].shape[-1]:]
    bot_reply = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    print(f"Bot: {bot_reply}\n")

    # 6. Save assistant reply to history so the model remembers context on the next turn!
    chat_history.append({"role": "assistant", "content": bot_reply})

🤖 Local LLM Chatbot Ready! (Type 'exit' or 'quit' to stop)



You:  tell me more about NVIDIA Scandal 


Bot: The NVIDIA scandal is a global phenomenon that has left a lasting impact on the technology industry. It started with a public statement from NVIDIA CEO Terrence Fernandez, stating that the company had "improved its GPU technology significantly." However, the news quickly turned ugly when the company's own security team discovered that some of the proprietary features used in its GPUs were being exploited by malicious actors.

The scandal has been ongoing since the beginning, with several companies and individuals publicly accusing NVIDIA of being responsible for the breach. Some of the key players involved include Google, Microsoft,



You:  what is Options Trading? Explain covered calls advantages


Bot: Options Trading is a financial instrument that allows investors to buy and sell options contracts on a specific stock, option, or other security. The primary advantage of options Trading is that it provides a liquidity hedge against potential losses.

In Options Trading, the underlying stock is the underlying asset, such as a stock, bond, or index. When an investor buys an Option, they are essentially buying a right to buy the underlying asset at a certain price at a future point in time. The option is bought in a short-term market, which means the price of the option is not fixed but is determined

